In [1]:
import numpy as np
import lightgbm as lgb
import catboost
import xgboost
import optuna

print("NumPy:", np.__version__)
print("LightGBM:", lgb.__version__)
print("CatBoost:", catboost.__version__)
print("XGBoost:", xgboost.__version__)
print("Optuna:", optuna.__version__)

print("Ready to train!")

NumPy: 2.4.6
LightGBM: 4.7.0
CatBoost: 1.2.10
XGBoost: 3.4.1
Optuna: 5.0.0
Ready to train!


In [2]:
import os, warnings, time
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from scipy.optimize import minimize
from scipy.special import logit, expit
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.preprocessing import QuantileTransformer
import keras
from keras import layers

warnings.filterwarnings("ignore")
SEED = 50
np.random.seed(SEED)
print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "| keras", keras.__version__, keras.backend.backend())

lightgbm 4.7.0 | xgboost 3.4.1 | keras 3.15.1 tensorflow


In [3]:
train = pd.read_csv(r"C:\Users\23108\Downloads\mobile money project\Train.csv")
test = pd.read_csv(r"C:\Users\23108\Downloads\mobile money project\Test (1).csv")
TARGET, ID_COL = "liquidity_stress_next_30d", "ID"
CATS = ["gender", "region", "smartphone", "segment", "earning_pattern"]
y = train[TARGET].values
print(train.shape, test.shape, "positive rate:", y.mean().round(4))

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\23108\\Downloads\\mobile money project\\Test (1).csv'

## Feature engineering
M1 is the most recent month and M6 the oldest. Inflow channels: P2P received, cash deposit, bank transfer in.
Outflow channels: P2P send, cash withdrawal, paybill, merchant pay.

In [ ]:
CH_IN = ["received", "deposit", "transfer_from_bank"]
CH_OUT = ["mm_send", "withdraw", "paybill", "merchantpay"]
CH = CH_IN + CH_OUT
UNIQ = {"paybill": "companies", "merchantpay": "merchants", "transfer_from_bank": "banks",
        "mm_send": "recipients", "received": "senders", "deposit": "agents", "withdraw": "agents"}
MONTHS = range(1, 7)
TC = np.arange(1, 7)[::-1].astype(float); TC = TC - TC.mean()   # time index, oldest -> newest
E = 1.0  # smoothing constant for ratios


def slope(A):
    """OLS slope over the 6 months (columns ordered m1..m6), positive = rising toward the present."""
    return (A * TC).sum(1) / (TC ** 2).sum()


def mat(df, fmt):
    return np.column_stack([df[fmt.format(m)].values.astype(float) for m in MONTHS])


def build_features(df):
    o = {}
    B = mat(df, "m{}_daily_avg_bal")
    bal_mean, bal_min = B.mean(1), B.min(1)
    o["bal_mean"] = bal_mean; o["bal_std"] = B.std(1); o["bal_min"] = bal_min; o["bal_max"] = B.max(1)
    o["bal_slope"] = slope(B); o["bal_slope_rel"] = o["bal_slope"] / (bal_mean + E)
    o["bal_cv"] = o["bal_std"] / (bal_mean + E)
    o["bal_m1_m6"] = B[:, 0] / (B[:, 5] + E); o["bal_m1_m2"] = B[:, 0] / (B[:, 1] + E)
    o["bal_m1_mean"] = B[:, 0] / (bal_mean + E); o["bal_m13_m46"] = B[:, :3].mean(1) / (B[:, 3:].mean(1) + E)
    o["bal_argmin"] = B.argmin(1); o["bal_argmax"] = B.argmax(1)
    o["bal_ndecl"] = (np.diff(B[:, ::-1], axis=1) < 0).sum(1)
    o["bal_m1_min"] = B[:, 0] / (bal_min + E)
    for k in range(5):
        o[f"bal_d{k+1}"] = B[:, k] - B[:, k + 1]

    V, TV = {}, {}
    for c in CH:
        v = V[c] = mat(df, "m{}_" + c + "_volume")
        tv = TV[c] = mat(df, "m{}_" + c + "_total_value")
        H = mat(df, "m{}_" + c + "_highest_amount")
        U = mat(df, "m{}_" + c + "_" + UNIQ[c])
        o[f"{c}_tv_mean"] = tv.mean(1); o[f"{c}_tv_std"] = tv.std(1); o[f"{c}_tv_slope"] = slope(tv) / (tv.mean(1) + E)
        o[f"{c}_tv_m1_mean"] = tv[:, 0] / (tv.mean(1) + E); o[f"{c}_tv_m13_m46"] = tv[:, :3].sum(1) / (tv[:, 3:].sum(1) + E)
        o[f"{c}_v_mean"] = v.mean(1); o[f"{c}_v_slope"] = slope(v) / (v.mean(1) + E); o[f"{c}_v_m1_mean"] = v[:, 0] / (v.mean(1) + E)
        o[f"{c}_zero_m"] = (v == 0).sum(1); o[f"{c}_last_active"] = np.where((v > 0).any(1), (v > 0).argmax(1), 6)
        o[f"{c}_avg_size"] = tv.sum(1) / (v.sum(1) + E); o[f"{c}_avg_size_m1"] = tv[:, 0] / (v[:, 0] + E)
        o[f"{c}_hi_share"] = H.sum(1) / (tv.sum(1) + E); o[f"{c}_hi_max"] = H.max(1)
        o[f"{c}_uniq_mean"] = U.mean(1); o[f"{c}_uniq_m1_mean"] = U[:, 0] / (U.mean(1) + E)
        o[f"{c}_tv_to_bal"] = tv.mean(1) / (bal_mean + E)

    IN = sum(TV[c] for c in CH_IN); OUT = sum(TV[c] for c in CH_OUT); NET = IN - OUT
    for k in range(6):
        o[f"net_m{k+1}"] = NET[:, k]; o[f"io_ratio_m{k+1}"] = OUT[:, k] / (IN[:, k] + E)
    in_mean, out_mean = IN.mean(1), OUT.mean(1)
    o["in_mean"] = in_mean; o["out_mean"] = out_mean; o["net_sum"] = NET.sum(1); o["net_m13"] = NET[:, :3].sum(1)
    o["io_ratio"] = OUT.sum(1) / (IN.sum(1) + E); o["io_ratio_m13"] = OUT[:, :3].sum(1) / (IN[:, :3].sum(1) + E)
    o["in_slope"] = slope(IN) / (in_mean + E); o["out_slope"] = slope(OUT) / (out_mean + E); o["net_slope"] = slope(NET)
    o["in_cv"] = IN.std(1) / (in_mean + E); o["out_cv"] = OUT.std(1) / (out_mean + E)
    o["coverage_m1"] = B[:, 0] / (OUT[:, 0] + E); o["coverage"] = bal_mean / (out_mean + E); o["bal_to_in"] = bal_mean / (in_mean + E)
    o["bal_m1_to_out_mean"] = B[:, 0] / (out_mean + E)
    o["withdraw_share_in"] = TV["withdraw"].sum(1) / (IN.sum(1) + E)
    o["withdraw_share_out"] = TV["withdraw"].sum(1) / (OUT.sum(1) + E)
    o["send_share_out"] = TV["mm_send"].sum(1) / (OUT.sum(1) + E)
    o["bill_share_out"] = (TV["paybill"] + TV["merchantpay"]).sum(1) / (OUT.sum(1) + E)
    o["recv_share_in"] = TV["received"].sum(1) / (IN.sum(1) + E); o["bank_share_in"] = TV["transfer_from_bank"].sum(1) / (IN.sum(1) + E)
    o["dep_share_in"] = TV["deposit"].sum(1) / (IN.sum(1) + E)
    VT = sum(V[c] for c in CH)
    o["vol_total"] = VT.sum(1); o["vol_slope"] = slope(VT) / (VT.mean(1) + E); o["vol_m1_mean"] = VT[:, 0] / (VT.mean(1) + E)
    arpu = df["arpu"].values
    o["arpu_to_bal"] = arpu / (bal_mean + E); o["arpu_to_in"] = arpu / (in_mean + E); o["arpu_to_out"] = arpu / (out_mean + E)
    R = np.column_stack([(B[:, k] - B[:, k + 1]) - NET[:, k] for k in range(5)])   # balance change not explained by flows
    for k in range(5):
        o[f"resid_m{k+1}"] = R[:, k]
    o["resid_abs_mean"] = np.abs(R).mean(1)

    # Window contrasts: recent s months vs the remaining months, for every series
    S = {"bal": B, "in": IN, "out": OUT, "vt": VT}
    for c in CH:
        S[c + "_tv"] = TV[c]; S[c + "_v"] = V[c]; S[c + "_u"] = mat(df, "m{}_" + c + "_" + UNIQ[c])
    for key, A in S.items():
        for s in (1, 2, 3):
            r, p = A[:, :s].mean(1), A[:, s:].mean(1)
            o[f"w{s}_{key}_pct"] = (r - p) / (np.abs(p) + E)
            if s != 3:
                o[f"w{s}_{key}_ratio"] = r / (p + E)
            o[f"w{s}_{key}_diff"] = r - p
    for s in (1, 2, 3):
        o[f"w{s}_net_diff"] = NET[:, :s].mean(1) - NET[:, s:].mean(1)
        o[f"w{s}_io_diff"] = OUT[:, :s].sum(1) / (IN[:, :s].sum(1) + E) - OUT[:, s:].sum(1) / (IN[:, s:].sum(1) + E)
    o["io_ratio_m46"] = OUT[:, 3:].sum(1) / (IN[:, 3:].sum(1) + E)
    o["net_m46"] = NET[:, 3:].sum(1)
    o["bal_m13_minus_m46_over_out"] = (B[:, :3].mean(1) - B[:, 3:].mean(1)) / (out_mean + E)
    return pd.DataFrame(o, index=df.index)

### Stress patterns, missed income and profile-relative features
Group medians come from the training rows only and never use the target.

In [ ]:
# New features: stress-pattern counts, missed-income flags, profile-relative scores.

def _run_len_from_start(M):
    """Length of the run of True values starting at column 0 (M1 backwards in time)."""
    out = np.zeros(len(M), int); alive = np.ones(len(M), bool)
    for k in range(M.shape[1]):
        alive &= M[:, k]; out += alive
    return out

def _longest_run(M):
    best = np.zeros(len(M), int); cur = np.zeros(len(M), int)
    for k in range(M.shape[1]):
        cur = np.where(M[:, k], cur + 1, 0); best = np.maximum(best, cur)
    return best

def build_new(df, ref):
    """df: raw rows to featurise; ref: raw TRAIN rows used for group medians (target not used)."""
    o = {}
    B = mat(df, "m{}_daily_avg_bal")
    TV = {c: mat(df, "m{}_" + c + "_total_value") for c in CH}
    V = {c: mat(df, "m{}_" + c + "_volume") for c in CH}
    IN = sum(TV[c] for c in CH_IN); OUT = sum(TV[c] for c in CH_OUT); NET = IN - OUT
    # --- stress-pattern counts
    o["n_out_gt_in"] = (NET < 0).sum(1)
    o["streak_out_gt_in"] = _run_len_from_start(NET < 0)          # consecutive deficit months ending at M1
    dec = B[:, :-1] < B[:, 1:]                                    # balance fell vs the month before (m_k < m_k+1)
    o["streak_bal_decline"] = _run_len_from_start(dec)
    o["longest_bal_decline"] = _longest_run(dec[:, ::-1])
    o["bal_m1_is_min"] = (B[:, 0] <= B.min(1)).astype(int)
    o["bal_m1_below_prev_min"] = (B[:, 0] < B[:, 1:].min(1)).astype(int)
    o["bal_m1_rank"] = (B[:, [0]] > B[:, 1:]).sum(1)
    o["bal_drawdown"] = (B.max(1) - B[:, 0]) / (B.max(1) + E)
    o["bal_m1_cover_prev_out"] = B[:, 0] / (OUT[:, 1:].mean(1) + E)
    cash_out = TV["withdraw"]
    o["cashout_share_m1"] = cash_out[:, 0] / (IN[:, 0] + E)
    o["cashout_share_mean"] = (cash_out / (IN + E)).mean(1)
    o["cashout_share_m1_vs_mean"] = o["cashout_share_m1"] - o["cashout_share_mean"]
    o["n_channels_m1"] = sum((V[c][:, 0] > 0).astype(int) for c in CH)
    o["n_channels_mean"] = sum((V[c] > 0).mean(1) for c in CH)
    o["channels_drop"] = o["n_channels_mean"] - o["n_channels_m1"]
    VT = sum(V[c] for c in CH)
    o["vol_m1_vs_m23"] = VT[:, 0] / (VT[:, 1:3].mean(1) + E)
    o["act_x_vol_drop"] = df["x_90_d_activity_rate"].values * o["vol_m1_vs_m23"]
    # --- missed expected income
    in_med = np.median(IN[:, 1:], 1)
    o["in_m1_vs_med"] = IN[:, 0] / (in_med + E)
    o["in_active_months"] = (IN > 0).sum(1)
    o["in_m1_zero"] = (IN[:, 0] == 0).astype(int)
    o["in_miss_m1"] = ((IN[:, 0] < 0.5 * in_med) & (in_med > 0)).astype(int)
    bank_act = (V["transfer_from_bank"][:, 1:] > 0).sum(1)
    o["bank_miss_m1"] = ((V["transfer_from_bank"][:, 0] == 0) & (bank_act >= 3)).astype(int)
    o["bank_m1_vs_med"] = TV["transfer_from_bank"][:, 0] / (np.median(TV["transfer_from_bank"][:, 1:], 1) + E)
    recv_act = (V["received"][:, 1:] > 0).sum(1)
    o["recv_miss_m1"] = ((V["received"][:, 0] == 0) & (recv_act >= 3)).astype(int)
    ep = df["earning_pattern"].values
    NA = -2.0   # "other earning pattern" code; real values are always > -1 (NaN slows XGBoost evaluation ~3x)
    for lab, tag in [("Monthly Earner", "mon"), ("Weekly Earner", "wk"), ("Irregular Earner", "irr")]:
        is_ = ep == lab
        o[f"in_m1_vs_med_{tag}"] = np.where(is_, o["in_m1_vs_med"], NA)
        o[f"in_miss_m1_{tag}"] = (is_ & (o["in_miss_m1"] == 1)).astype(int)
        o[f"w3_in_pct_{tag}"] = np.where(is_, (IN[:, :3].mean(1) - IN[:, 3:].mean(1)) / (IN[:, 3:].mean(1) + E), NA)
    # --- profile-relative scores (value / group median, groups from TRAIN rows)
    def keyvals(d):
        Bd = mat(d, "m{}_daily_avg_bal")
        INd = sum(mat(d, "m{}_" + c + "_total_value") for c in CH_IN)
        OUTd = sum(mat(d, "m{}_" + c + "_total_value") for c in CH_OUT)
        return pd.DataFrame({"bal_mean": Bd.mean(1), "bal_m1": Bd[:, 0], "in_mean": INd.mean(1), "in_m1": INd[:, 0],
                             "out_mean": OUTd.mean(1), "out_m1": OUTd[:, 0], "arpu": d["arpu"].values,
                             "io_m13": OUTd[:, :3].sum(1) / (INd[:, :3].sum(1) + E)}, index=d.index)
    Kd, Kr = keyvals(df), keyvals(ref)
    for grp in [["segment"], ["earning_pattern"], ["segment", "earning_pattern"], ["region"]]:
        g = "_".join(x[:3] for x in grp)
        kd = df[grp].astype(str).agg("|".join, axis=1).values
        kr = ref[grp].astype(str).agg("|".join, axis=1).values
        med = Kr.groupby(kr).median()
        for col in Kd.columns:
            m = pd.Series(kd).map(med[col]).values
            o[f"rel_{g}_{col}"] = Kd[col].values / (np.abs(m) + E)
    return pd.DataFrame(o, index=df.index)


RAW_TRAIN = train.drop(columns=[ID_COL, TARGET])


def make_matrix(df):
    X = df.drop(columns=[ID_COL, TARGET], errors="ignore").copy()
    out = pd.concat([X, build_features(X), build_new(X, RAW_TRAIN)], axis=1)
    for c in CATS:
        out[c] = pd.Categorical(out[c], categories=sorted(train[c].unique()))
    return out


t0 = time.time()
X_all, X_test_all = make_matrix(train), make_matrix(test)
print("feature matrix:", X_all.shape, X_test_all.shape, f"({time.time()-t0:.1f}s)")

## Feature selection: top 250 by LightGBM gain (first-pass 5-fold CV)

In [ ]:
skf42 = list(StratifiedKFold(10, shuffle=True, random_state=SEED).split(X_all, y))
fs_params = dict(n_estimators=4000, learning_rate=0.04, num_leaves=31, min_child_samples=50, subsample=0.8,
                 subsample_freq=1, colsample_bytree=0.3, reg_lambda=2, verbose=-1, random_state=SEED, n_jobs=-1)
gain, oof_fs = pd.Series(0.0, index=X_all.columns), np.zeros(len(y))
for tr_idx, va_idx in skf42:
    m = lgb.LGBMClassifier(**fs_params)
    m.fit(X_all.iloc[tr_idx], y[tr_idx], eval_set=[(X_all.iloc[va_idx], y[va_idx])],
          callbacks=[lgb.early_stopping(200, verbose=False)])
    oof_fs[va_idx] = m.predict_proba(X_all.iloc[va_idx])[:, 1]
    gain += pd.Series(m.booster_.feature_importance("gain"), index=X_all.columns)
print("first-pass LGB (all features)  LL %.5f  AUC %.5f" % (log_loss(y, oof_fs), roc_auc_score(y, oof_fs)))
FEATURES = list(gain.sort_values(ascending=False).index[:250])
X, X_test = X_all[FEATURES], X_test_all[FEATURES]
cat_in = [c for c in FEATURES if c in CATS]
print("selected:", len(FEATURES), "| categoricals kept:", cat_in)
print(gain.sort_values(ascending=False).head(15).round(0).to_string())

## Models (stratified 5-fold CV; test predictions = mean of fold models)

In [ ]:
def run_cv(name, fit_predict, seeds):
    oof_s, pt_s = [], []
    for sd in seeds:
        folds = StratifiedKFold(5, shuffle=True, random_state=sd).split(X, y)
        oof, pt = np.zeros(len(y)), np.zeros(len(X_test))
        for tr_idx, va_idx in folds:
            p_va, p_te = fit_predict(X.iloc[tr_idx], y[tr_idx], X.iloc[va_idx], y[va_idx], X_test, sd)
            oof[va_idx] = p_va; pt += p_te / 5
        print(f"  {name} seed {sd}: LL {log_loss(y, oof):.5f}  AUC {roc_auc_score(y, oof):.5f}", flush=True)
        oof_s.append(oof); pt_s.append(pt)
    oof, pt = np.mean(oof_s, 0), np.mean(pt_s, 0)
    print(f"{name} seed-avg: LL {log_loss(y, oof):.5f}  AUC {roc_auc_score(y, oof):.5f}")
    return oof, pt


def fp_lgb(Xa, ya, Xb, yb, Xt, sd):
    m = lgb.LGBMClassifier(n_estimators=6000, learning_rate=0.02, num_leaves=7, min_child_samples=100,
                           colsample_bytree=0.3, reg_lambda=5, subsample=0.8, subsample_freq=1,
                           verbose=-1, random_state=sd, n_jobs=-1)
    m.fit(Xa, ya, eval_set=[(Xb, yb)], callbacks=[lgb.early_stopping(300, verbose=False)])
    return m.predict_proba(Xb)[:, 1], m.predict_proba(Xt)[:, 1]


def fp_cb(Xa, ya, Xb, yb, Xt, sd):
    m = CatBoostClassifier(iterations=6000, learning_rate=0.04, depth=5, l2_leaf_reg=5, rsm=0.3,
                           loss_function="Logloss", random_seed=sd, od_type="Iter", od_wait=300,
                           verbose=0, thread_count=-1, cat_features=cat_in)
    m.fit(Xa, ya, eval_set=(Xb, yb), use_best_model=True)
    return m.predict_proba(Xb)[:, 1], m.predict_proba(Xt)[:, 1]


def fp_cb_ord(Xa, ya, Xb, yb, Xt, sd):
    # Same settings as fp_cb but with ordered boosting: slower, and its errors differ from plain boosting
    m = CatBoostClassifier(iterations=6000, learning_rate=0.04, depth=5, l2_leaf_reg=5, rsm=0.3, boosting_type="Ordered",
                           loss_function="Logloss", random_seed=sd, od_type="Iter", od_wait=300,
                           verbose=0, thread_count=-1, cat_features=cat_in)
    m.fit(Xa, ya, eval_set=(Xb, yb), use_best_model=True)
    return m.predict_proba(Xb)[:, 1], m.predict_proba(Xt)[:, 1]




SEEDS = [42, 7,2026, 123, 999]

t0 = time.time(); oof_lgb, pt_lgb = run_cv("LightGBM", fp_lgb, SEEDS); print(f"({(time.time()-t0)/60:.1f} min)")
t0 = time.time(); oof_cb, pt_cb = run_cv("CatBoost", fp_cb, SEEDS); print(f"({(time.time()-t0)/60:.1f} min)")
t0 = time.time(); oof_cbo, pt_cbo = run_cv("CatBoost-Ordered", fp_cb_ord, [42, 7]); print(f"({(time.time()-t0)/60:.1f} min)")


LightGBM seed-avg: LL 0.23228  AUC 0.91945
(2.5 min)


  CatBoost seed 42: LL 0.23214  AUC 0.91922


  CatBoost seed 7: LL 0.23202  AUC 0.91922


  CatBoost seed 2026: LL 0.23229  AUC 0.91915


CatBoost seed-avg: LL 0.23114  AUC 0.91994
(4.6 min)


  CatBoost-Ordered seed 42: LL 0.23156  AUC 0.91934


... (4 outputs left out)

  XGBoost seed 2026: LL 0.23347  AUC 0.91865


  XGBoost seed 123: LL 0.23350  AUC 0.91847


  XGBoost seed 999: LL 0.23396  AUC 0.91809


XGBoost seed-avg: LL 0.23276  AUC 0.91908
(12.0 min)


## Neural network: ReLU hidden layers, softmax output
Inputs: the 45 numeric features with the highest first-pass gain, quantile-transformed to a normal
distribution (fitted inside each training fold), plus one-hot profile categories.
Architecture: Dense(128, ReLU) -> BatchNorm -> Dropout(0.3) -> Dense(64, ReLU) -> Dropout(0.3) -> Dense(2, softmax).
Loss: categorical cross-entropy; Adam (lr 1e-3), batch 1024, early stopping on validation loss.
P(stress) is the second softmax unit.

In [ ]:
NN_NUM = [c for c in gain.sort_values(ascending=False).index if c not in CATS][:45]
D = pd.get_dummies(pd.concat([X_all[CATS], X_test_all[CATS]]).astype(str), dtype=float).values
D_tr, D_te = D[:len(X_all)], D[len(X_all):]
print("NN inputs:", len(NN_NUM), "numeric +", D.shape[1], "one-hot")


def build_nn(n_in):
    reg = keras.regularizers.l2(1e-5)
    inp = keras.Input((n_in,))
    h = layers.Dense(128, activation="relu", kernel_regularizer=reg)(inp)
    h = layers.BatchNormalization()(h)
    h = layers.Dropout(0.3)(h)
    h = layers.Dense(64, activation="relu", kernel_regularizer=reg)(h)
    h = layers.Dropout(0.3)(h)
    out = layers.Dense(2, activation="softmax")(h)
    m = keras.Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy")
    return m


def run_nn(seeds):
    oof_s, pt_s = [], []
    for sd in seeds:
        oof, pt = np.zeros(len(y)), np.zeros(len(X_test_all))
        for tr_idx, va_idx in StratifiedKFold(5, shuffle=True, random_state=sd).split(X_all, y):
            qt = QuantileTransformer(n_quantiles=1000, output_distribution="normal", random_state=sd)
            Xa = np.hstack([np.nan_to_num(qt.fit_transform(X_all[NN_NUM].iloc[tr_idx])), D_tr[tr_idx]])
            Xb = np.hstack([np.nan_to_num(qt.transform(X_all[NN_NUM].iloc[va_idx])), D_tr[va_idx]])
            Xt = np.hstack([np.nan_to_num(qt.transform(X_test_all[NN_NUM])), D_te])
            keras.utils.set_random_seed(sd)
            m = build_nn(Xa.shape[1])
            m.fit(Xa, y[tr_idx], validation_data=(Xb, y[va_idx]), epochs=200, batch_size=1024, verbose=0,
                  callbacks=[keras.callbacks.EarlyStopping(patience=12, restore_best_weights=True),
                             keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=4, min_lr=1e-5)])
            oof[va_idx] = m.predict(Xb, verbose=0, batch_size=4096)[:, 1]
            pt += m.predict(Xt, verbose=0, batch_size=4096)[:, 1] / 5
        print(f"  NN seed {sd}: LL {log_loss(y, oof):.5f}  AUC {roc_auc_score(y, oof):.5f}", flush=True)
        oof_s.append(oof); pt_s.append(pt)
    oof, pt = np.mean(oof_s, 0), np.mean(pt_s, 0)
    print(f"NN seed-avg: LL {log_loss(y, oof):.5f}  AUC {roc_auc_score(y, oof):.5f}")
    return oof, pt


t0 = time.time(); oof_nn, pt_nn = run_nn([42, 7, 2026, 123, 999]); print(f"({(time.time()-t0)/60:.1f} min)")

  NN seed 2026: LL 0.24654  AUC 0.90837


  NN seed 123: LL 0.24808  AUC 0.90646


  NN seed 999: LL 0.24698  AUC 0.90772


NN seed-avg: LL 0.24338  AUC 0.91061
(5.8 min)


## Blend in logit space (non-negative weights summing to 1, fitted to OOF log loss)

In [ ]:
names = ["lgb", "cb", "cb_ord", "nn"]
OOF = np.column_stack([oof_lgb, oof_cb, oof_cbo, oof_nn]); PT = np.column_stack([pt_lgb, pt_cb, pt_cbo, pt_nn])
Lo, Lt = logit(np.clip(OOF, 1e-6, 1 - 1e-6)), logit(np.clip(PT, 1e-6, 1 - 1e-6))
print("OOF correlation:\n", pd.DataFrame(np.corrcoef(OOF.T), index=names, columns=names).round(4))

def norm(w): w = np.abs(w); return w / w.sum()
res = minimize(lambda w: log_loss(y, expit(Lo @ norm(w))), np.ones(len(names)) / len(names), method="Nelder-Mead", options={"maxiter": 4000})
W = norm(res.x)
oof_blend, test_pred = expit(Lo @ W), expit(Lt @ W)
print("weights:", dict(zip(names, W.round(3))))
print("Blend OOF  LL %.5f  AUC %.5f" % (log_loss(y, oof_blend), roc_auc_score(y, oof_blend)))
print("Baselines  prior-only LL %.5f" % log_loss(y, np.full(len(y), y.mean())))

# calibration check: mean predicted vs observed by decile
cal = pd.DataFrame({"p": oof_blend, "y": y}); cal["bin"] = pd.qcut(cal.p, 10, labels=False)
print(cal.groupby("bin").agg(mean_pred=("p", "mean"), observed=("y", "mean"), n=("y", "size")).round(3).to_string())

## Submission (raw probabilities)

In [ ]:
submission = pd.DataFrame({"ID": test[ID_COL], "Target": test_pred})
assert submission.shape[0] == test.shape[0] and submission.Target.between(0, 1).all() and submission.ID.is_unique
submission.to_csv("submissionv1.csv", index=False)
print(submission.shape, "| mean pred %.4f (train rate %.4f)" % (test_pred.mean(), y.mean()))
submission.head()